# Link a PDF score to an MEI file, step by step

**Workflow 1 — editing music with the MEI data format. Guided version for beginners.** This notebook connects a scanned score (a PDF) with its MEI encoding, bar by bar. Afterwards you can point at a bar in the encoded score and see the same bar highlighted in the scan.

| Step | What the computer does | What you do |
| --- | --- | --- |
| 1. PDF → page images | turns every PDF page into a picture | look at the pictures |
| 2. Describe the pages | checks your description against the MEI | write one line per page with music |
| 3. Find the bars | a detector draws a box around every bar | check the boxes, add one the detector missed |
| 4. Merge with the MEI | writes the boxes into a copy of your MEI file | run the cell |
| 5. Look at the result | shows score and scan side by side | point and click |

The example is Dieterich Buxtehude's Suite from *Denkmäler deutscher Tonkunst*, vol. 11 (Leipzig 1903), pp. 160–163: 89 bars on four pages. The file `test_corpus/Buxtehude-DdT11-app01-suite.mei` holds its encoding. Your original MEI file is never changed; everything new goes into the folder `converted_mei/pdf_guided/`.

For several movements in one file, schema checks and more control, use the full version: [`mei_pdf_facsimile_integration.ipynb`](mei_pdf_facsimile_integration.ipynb).

## Before you start: how Jupyter works

- A notebook is made of **cells**. Grey cells contain Python code; the others contain text like this one.
- To **run** a cell, click into it and press **Shift + Enter** (or click ▶ in the toolbar). While it runs, `[*]` appears next to it; when it has finished, a number appears.
- Run the cells **from top to bottom**. Every step needs the steps before it.
- Change only the cells marked **✏️**, and in them only the text between quotes `"…"`, the numbers, and `True` / `False`. Keep the quotes, commas and brackets.
- Lines that start with `#` are comments. Python ignores them; they explain the line below.
- When something is missing, the cell prints **⚠ NOT DONE** and says what to do. To start again from a clean state, use **Kernel → Restart Kernel and Run All Cells** (JupyterLab; other programs have a similar menu entry). Your files stay where they are.

## Setup

Run the next two cells. The first loads CAMAT. The second loads the helper code of this notebook; its text is folded away because you do not need to read it.

In [ ]:
# Cloud Jupyter: install CAMAT and copy notebooks/test_corpus if they are not already here.
try:
    import setup_camat
except ModuleNotFoundError:
    pass
try:
    from camat.notebook_workspace import prepare_notebook
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "camat"])
    from camat.notebook_workspace import prepare_notebook

prepare_notebook()

# You can leave this cell unchanged.
import setup_camat

try:
    from camat import pdf_facsimile
except ImportError as problem:
    raise ImportError(
        "This notebook needs a CAMAT version with PDF support (camat.pdf_facsimile). "
        "Update CAMAT with: python -m pip install -U camat"
    ) from problem
print("✓ CAMAT is ready.")

In [ ]:
# Helper code for this notebook. Run it; you do not need to read or change it.
# It calls camat.pdf_facsimile and turns its messages into plain language.
import json
import re
import tempfile
import traceback
import urllib.request
import warnings
import xml.etree.ElementTree as ET
from copy import deepcopy
from datetime import date
from functools import wraps
from pathlib import Path

from PIL import Image as PILImage, ImageDraw, ImageFont

from camat import display_path, launch_interactive_facsimile_viewer, resolve_repo_path
from camat.pdf_facsimile import (
    _ordered_systems,  # groups boxes into systems exactly like the check in step 3b
    build_measure_alignment,
    detect_measure_boxes,
    draw_alignment_overlays,
    integrate_pdf_facsimile,
    render_pdf_pages,
    write_page_breaks,
)

EXAMPLE_PDF = "converted_mei/pdf_guided/bsb00023199_0170-0173.pdf"
BSB_FORM_URL = (
    "https://download.digitale-sammlungen.de/BOOKS/download.pl"
    "?id=00023199&ersteseite=170&letzteseite=173&xdfz=2&vers=d&nr=&abschicken=ja"
    "&captcha_zeit=00023199"
)
BSB_HOST = "https://download.digitale-sammlungen.de"
MEI_NS = "{http://www.music-encoding.org/ns/mei}"
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"
OTHER_PAGE_TYPES = ("cover", "title", "preface", "blank", "advertisement", "duplicate")
COORDS = ("ulx", "uly", "lrx", "lry")


class NotDone(Exception):
    """A step stops here; the message says what to do."""


def _step(function):
    """Print a short NOT DONE message instead of a long error report."""
    @wraps(function)
    def run(*args, **kwargs):
        try:
            return function(*args, **kwargs)
        except NotDone as problem:
            print(f"\n⚠ NOT DONE. {problem}")
        except Exception as problem:
            print(f"\n⚠ NOT DONE. Something unexpected happened: {type(problem).__name__}: {problem}")
            print("Show this message and the details below to your teacher.")
            traceback.print_exc()
        return None
    return run


def _folders(output_folder):
    out = resolve_repo_path(output_folder)
    pages = out / "pages"
    return {"out": out, "pages": pages, "pages_json": pages / "pages.json", "boxes": pages / "boxes",
            "alignment": pages / "alignment.json", "overlays": out / "overlays",
            "previews": out / "previews", "mei": out / "mei"}


def _switch(value, name):
    if not isinstance(value, bool):
        raise NotDone(f"{name} must be True or False, written without quotes. "
                      "Correct it in the settings cell and run that cell again.")
    return value


def _in_jupyter():
    try:
        from IPython import get_ipython
    except ImportError:
        return False
    shell = get_ipython()
    return shell is not None and getattr(shell, "kernel", None) is not None


def _show_file(path, width=None):
    if _in_jupyter():
        from IPython.display import Image, display
        display(Image(filename=str(path), width=width))
    else:
        print(f"(picture: {display_path(path)})")


def _show(image, path, width=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    image.save(path)
    _show_file(path, width)


def _font(size):
    try:
        return ImageFont.truetype("DejaVuSans.ttf", size)
    except OSError:
        return ImageFont.load_default(size=size)


def _sheet(tiles, columns, tile_width):
    """Put (picture, caption) pairs side by side."""
    thumbs = []
    for image, caption in tiles:
        height = round(image.height * tile_width / image.width)
        thumbs.append((image.convert("RGB").resize((tile_width, height)), caption))
    cell_height = max(thumb.height for thumb, _ in thumbs) + 40
    rows = (len(thumbs) + columns - 1) // columns
    sheet = PILImage.new("RGB", (min(columns, len(thumbs)) * (tile_width + 20) + 20, rows * cell_height + 20), "white")
    draw, font = ImageDraw.Draw(sheet), _font(18)
    for index, (thumb, caption) in enumerate(thumbs):
        x = 20 + (index % columns) * (tile_width + 20)
        y = 20 + (index // columns) * cell_height
        draw.text((x, y), caption, fill="black", font=font)
        sheet.paste(thumb, (x, y + 28))
        draw.rectangle([x - 1, y + 27, x + thumb.width, y + 28 + thumb.height], outline=(150, 150, 150))
    return sheet


def _ranges(labels):
    """['1', '2', '3', '7'] -> '1–3, 7'"""
    parts, run = [], []
    for label in labels:
        if run and label.isdigit() and run[-1].isdigit() and int(label) == int(run[-1]) + 1:
            run.append(label)
            continue
        if run:
            parts.append(run[0] if len(run) == 1 else f"{run[0]}–{run[-1]}")
        run = [label]
    if run:
        parts.append(run[0] if len(run) == 1 else f"{run[0]}–{run[-1]}")
    return ", ".join(parts)


def _read_pages(folders):
    if not folders["pages_json"].is_file():
        raise NotDone("There are no page images yet. Run step 1 first.")
    return json.loads(folders["pages_json"].read_text(encoding="utf-8"))


def _music_pages(folders):
    music = [page for page in _read_pages(folders)["pages"] if page.get("role") == "music"]
    if not music:
        raise NotDone("No page is described as music yet. Run step 2 first.")
    return music


def _page_name(page):
    printed = page.get("printed_page")
    return f"PDF page {page['pdf_page_number']}" + (f" (printed page {printed})" if printed else "")


# ---------- Step 1 ----------

def _download_example(pdf):
    print("Downloading the example PDF from the Bayerische Staatsbibliothek ...")
    with urllib.request.urlopen(BSB_FORM_URL, timeout=120) as response:
        html = response.read().decode("utf-8", errors="replace")
    links = re.findall(r'href=[\'"](/pdf/\d+bsb00023199\.pdf)[\'"]', html)
    if len(links) != 1:
        raise NotDone("The library's download page looked different than expected. "
                      "Download the PDF by hand (see the instructions above).")
    with urllib.request.urlopen(BSB_HOST + links[0], timeout=300) as response:
        data = response.read()
    if not data.startswith(b"%PDF-"):
        raise NotDone("The library did not send a PDF. Download it by hand (see the instructions above).")
    pdf.parent.mkdir(parents=True, exist_ok=True)
    pdf.write_bytes(data)
    print(f"Saved {display_path(pdf)} ({len(data) // 1024} KB). "
          "Licence: CC BY-NC-SA 4.0, Bayerische Staatsbibliothek. Keep this file.")


def _require_poppler():
    import shutil
    if shutil.which("pdftoppm") and shutil.which("pdfinfo"):
        return
    raise NotDone(
        "This step needs the free program Poppler, which is not installed.\n"
        "  Windows / any system with conda:  conda install -c conda-forge poppler\n"
        "  macOS with Homebrew:              brew install poppler\n"
        "  Debian / Ubuntu:                  sudo apt install poppler-utils\n"
        "  Google Colab:                     run  !apt-get install -y poppler-utils  in a new cell\n"
        "Then restart the kernel (Kernel → Restart Kernel) and run the notebook again from the top.")


@_step
def make_page_images(pdf_file, output_folder, *, download_example=False):
    """Step 1: turn every PDF page into a PNG picture and show small previews."""
    download_example = _switch(download_example, "DOWNLOAD_EXAMPLE_PDF")
    folders, pdf = _folders(output_folder), resolve_repo_path(pdf_file)
    if not pdf.is_file():
        if download_example and pdf == resolve_repo_path(EXAMPLE_PDF):
            _download_example(pdf)
        elif download_example:
            raise NotDone(f"DOWNLOAD_EXAMPLE_PDF only fetches the Buxtehude example. "
                          f"Your PDF_FILE points to {display_path(pdf)}, which does not exist. "
                          "Put your PDF there, or correct PDF_FILE in the settings cell.")
        else:
            raise NotDone(f"The PDF is missing: {display_path(pdf)}\n"
                          "For the example, set DOWNLOAD_EXAMPLE_PDF = True in the settings cell (and run it), "
                          "or download the PDF by hand as described above.")
    _require_poppler()
    source_id = re.sub(r"[^A-Za-z0-9_.-]", "_", pdf.stem)
    if not re.match(r"[A-Za-z_]", source_id):
        source_id = "pdf_" + source_id
    print("Turning the PDF pages into pictures (300 dpi). This takes a moment ...")
    try:
        manifest = render_pdf_pages(pdf, folders["pages"], dpi=300, source_id=source_id)
    except (ValueError, FileExistsError) as problem:
        raise NotDone(f"The folder {display_path(folders['pages'])} already holds pictures of another PDF, "
                      "or of an earlier download of this one (every download counts as a new file). "
                      "Set a new OUTPUT_FOLDER in the settings cell, run that cell, and start again with step 1.\n"
                      f"(Details: {problem})")
    tiles = []
    for page in manifest["pages"]:
        with PILImage.open(folders["pages"] / page["image"]) as image:
            tiles.append((image.copy(), f"PDF page {page['pdf_page_number']}"))
    print(f"✓ {manifest['page_count']} pages, saved in {display_path(folders['pages'])}")
    _show(_sheet(tiles, columns=5, tile_width=220), folders["previews"] / "pages.png")
    print("Which PDF pages show music? You need that for step 2.")


# ---------- Step 2 ----------

def _labels(text, line_no, shown):
    """'1-4 6' -> ['1', '2', '3', '4', '6']"""
    text = re.sub(r"\s*[-–—]\s*", "-", text.strip())
    if not text:
        raise NotDone(f"Line {line_no} ({shown}): one system is empty. Are there two commas in a row?")
    labels = []
    for token in text.split():
        match = re.fullmatch(r"(\d+)-(\d+)", token)
        if match:
            first, last = int(match[1]), int(match[2])
            if first > last:
                raise NotDone(f"Line {line_no} ({shown}): {token} counts backwards. Write the smaller number first.")
            labels.extend(str(n) for n in range(first, last + 1))
        elif "-" in token:
            raise NotDone(f"Line {line_no} ({shown}): I cannot read {token!r}. Write a range like 5-9.")
        else:
            labels.append(token)
    return labels


def _parse_description(text, page_count):
    pages = {}
    for line_no, raw in enumerate(text.splitlines(), 1):
        line, shown = raw.split("#", 1)[0].strip(), raw.strip()
        if not line:
            continue
        parts = [part.strip() for part in line.split("|")]
        if not parts[0].isdigit():
            raise NotDone(f"Line {line_no} ({shown}) must start with a PDF page number, "
                          "for example: 2 | 160 | 1-4, 5-9, 10-14")
        number = int(parts[0])
        if not 1 <= number <= page_count:
            raise NotDone(f"Line {line_no} ({shown}): the PDF only has pages 1 to {page_count}.")
        if number in pages:
            raise NotDone(f"Line {line_no} ({shown}): PDF page {number} is described twice.")
        if len(parts) == 2:
            kind = parts[1].lower()
            if kind not in OTHER_PAGE_TYPES:
                raise NotDone(f"Line {line_no} ({shown}): a page with music needs three parts, "
                              "PDF page | printed page | bars. For a page without music, use one of: "
                              + ", ".join(OTHER_PAGE_TYPES) + ".")
            pages[number] = {"role": kind}
        elif len(parts) == 3:
            printed = None if parts[1] in ("", "-", "?") else parts[1]
            systems = [_labels(system, line_no, shown) for system in parts[2].split(",")]
            pages[number] = {"role": "music", "printed_page": printed, "systems": systems}
        else:
            raise NotDone(f"Line {line_no} ({shown}): expected 2 or 3 parts separated by |, found {len(parts)}.")
    if not any(page["role"] == "music" for page in pages.values()):
        raise NotDone("Your description has no page with music. "
                      "Write one line per music page, for example: 2 | 160 | 1-4, 5-9, 10-14")
    return pages


def _mei_movement(mei):
    """Return (mdiv/@n or None, bar numbers in order) of the file's only movement."""
    if not mei.is_file():
        raise NotDone(f"The MEI file is missing: {display_path(mei)}. Check MEI_FILE in the settings cell.")
    try:
        root = ET.parse(mei).getroot()
    except ET.ParseError as problem:
        raise NotDone(f"{display_path(mei)} is not a readable XML file: {problem}")
    mdivs = list(root.iter(MEI_NS + "mdiv"))
    if not mdivs:
        raise NotDone(f"{display_path(mei)} has no <mdiv> (movement). Is it an MEI file?")
    if len(mdivs) > 1:
        raise NotDone(f"{display_path(mei)} has {len(mdivs)} movements (<mdiv> elements). This guided notebook "
                      "links one movement. For several, use mei_pdf_facsimile_integration.ipynb.")
    bars = []
    for measure in mdivs[0].iter(MEI_NS + "measure"):
        if measure.get("n") is None:
            raise NotDone(f"The MEI has a <measure> without a bar number (@n), xml:id {measure.get(XML_ID)}. "
                          "Every bar needs a number.")
        bars.append(measure.get("n"))
    if not bars:
        raise NotDone(f"{display_path(mei)} has no <measure> elements.")
    return mdivs[0].get("n"), bars


def _split_bars(pages):
    """Bar numbers written at the end of one system and again at the start of the next."""
    previous, split = None, []
    for number in sorted(pages):
        for system in pages[number].get("systems", []):
            if previous is not None and system[0] == previous:
                split.append(system[0])
            previous = system[-1]
    return split


def _checked_description(description, mei_file, manifest):
    mdiv_n, bars = _mei_movement(resolve_repo_path(mei_file))
    pages = _parse_description(description, manifest["page_count"])
    sequence = []
    for number in sorted(pages):
        for system in pages[number].get("systems", []):
            sequence.extend(system[1:] if sequence and system[0] == sequence[-1] else system)
    missing = [bar for bar in bars if bar not in set(sequence)]
    extra = [label for label in sequence if label not in set(bars)]
    if missing or extra:
        lines = ["Your description and the MEI do not have the same bars."]
        if missing:
            lines.append(f"  In the MEI but not in your description: {_ranges(missing)}")
        if extra:
            lines.append(f"  In your description but not in the MEI: {_ranges(extra)}")
        lines.append(f"The MEI has {len(bars)} bars: {_ranges(bars)}. Correct the description and run this cell again.")
        raise NotDone("\n".join(lines))
    if sequence != bars:
        index = next((i for i, (a, b) in enumerate(zip(sequence, bars)) if a != b), min(len(sequence), len(bars)))
        before = f"after bar {bars[index - 1]}" if index else "at the start"
        raise NotDone(f"Your bars are not in the same order as in the MEI: {before}, the MEI continues with "
                      f"bar {bars[index] if index < len(bars) else '(end)'}, your description with "
                      f"bar {sequence[index] if index < len(sequence) else '(end)'}. "
                      "Is a bar listed twice or in the wrong place?")
    return pages, mdiv_n, len(bars)


def _forget_description(folders, manifest):
    """After a failed step 2, later steps must not use an older description."""
    if not any(page.get("role") == "music" for page in manifest["pages"]):
        return
    for page in manifest["pages"]:
        page.pop("mdiv", None)
        page.pop("systems", None)
        page.update(role="unreviewed", printed_page=None)
    folders["pages_json"].write_text(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")


@_step
def describe_pages(description, mei_file, output_folder):
    """Step 2: check the page description against the MEI and save it in pages.json."""
    folders = _folders(output_folder)
    manifest = _read_pages(folders)
    try:
        pages, mdiv_n, bar_count = _checked_description(description, mei_file, manifest)
    except NotDone:
        _forget_description(folders, manifest)
        raise
    split = _split_bars(pages)

    for page in manifest["pages"]:
        entry = pages.get(page["pdf_page_number"], {"role": "unreviewed"})
        page.pop("mdiv", None)
        page.pop("systems", None)
        page["role"] = entry["role"]
        page["printed_page"] = entry.get("printed_page")
        if entry["role"] == "music":
            page["mdiv"] = mdiv_n or "1"
            page["systems"] = entry["systems"]
    text = json.dumps(manifest, indent=2, ensure_ascii=False) + "\n"
    if folders["pages_json"].read_text(encoding="utf-8") != text:
        folders["pages_json"].write_text(text, encoding="utf-8")

    print("PDF page | printed page | content")
    for page in manifest["pages"]:
        if page["role"] == "music":
            labels = [label for system in page["systems"] for label in system]
            content = f"music: {len(page['systems'])} systems, bars {_ranges(list(dict.fromkeys(labels)))}"
        elif page["role"] == "unreviewed":
            content = "not described, so no music"
        else:
            content = f"{page['role']}, no music"
        print(f"{page['pdf_page_number']:>8} | {page['printed_page'] or '-':>12} | {content}")
    print(f"\n✓ Your description matches the MEI: all {bar_count} bars, in the same order.")
    if split:
        print(f"  Bars that continue on the next system: {', '.join(split)}")
    if mdiv_n is None:
        print("  (The movement in the MEI has no number; step 4 numbers it 1 in the new file.)")


# ---------- Step 3 ----------

def _box_files(folders, page):
    path = folders["boxes"] / f"{page['page_id']}.json"
    return path, path.with_name(f"{page['page_id']}.original.json")


def _boxes(folders, page):
    """Return (the detector's boxes, the boxes in use now)."""
    path, original = _box_files(folders, page)
    current = json.loads(path.read_text(encoding="utf-8"))
    first = json.loads(original.read_text(encoding="utf-8")) if original.is_file() else current
    return first, current


def _labelled_page(folders, page, boxes, labels):
    with PILImage.open(folders["pages"] / page["image"]) as source:
        image = source.convert("RGB")
    draw, font = ImageDraw.Draw(image, "RGBA"), _font(max(24, image.width // 32))
    for box, label in zip(boxes, labels):
        color = (30, 90, 220) if label == "new" else (210, 30, 30)
        draw.rectangle([box[k] for k in COORDS], outline=(*color, 255), width=max(3, image.width // 400))
        x, y = box["ulx"] + 8, box["uly"] + 8
        draw.rectangle(draw.textbbox((x, y), label, font=font), fill=(255, 255, 255, 220))
        draw.text((x, y), label, font=font, fill=(*color, 255))
    return image


def _labels_for(original, boxes):
    return [str(original["measures"].index(box)) if box in original["measures"] else "new" for box in boxes]


@_step
def find_bars(output_folder, *, send_to_detector=False):
    """Step 3a: get one box per bar from the measure detector (or reuse saved boxes)."""
    send_to_detector = _switch(send_to_detector, "SEND_PAGES_TO_DETECTOR")
    folders = _folders(output_folder)
    music = _music_pages(folders)
    missing = [page for page in music if not _box_files(folders, page)[0].is_file()]
    if missing and not send_to_detector:
        raise NotDone("The detector has not seen these pages yet: "
                      + ", ".join(_page_name(page) for page in missing) + ".\n"
                      "Set SEND_PAGES_TO_DETECTOR = True in the settings cell, run that cell, "
                      "then run this cell again.")
    if missing:
        print(f"Sending {len(missing)} page(s) to the measure detector. This can take a minute per page ...")
    try:
        records = detect_measure_boxes(folders["pages_json"], allow_upload=send_to_detector)
    except (RuntimeError, OSError) as problem:
        raise NotDone(f"The detector could not be reached or did not answer: {problem}\n"
                      "Check the internet connection and run this cell again later.")
    except ValueError as problem:
        if str(problem).startswith("Detector response"):
            raise NotDone(f"The detector sent an answer CAMAT cannot read ({problem}). Try again later.")
        raise NotDone(f"The saved boxes do not fit the page pictures ({problem}). "
                      "Set a new OUTPUT_FOLDER in the settings cell and start again with step 1.")
    sources = {record["page_id"]: record["source"] for record in records}
    tiles = []
    for page in music:
        original, current = _boxes(folders, page)
        boxes = original["measures"]
        note = " (in use: your corrections from step 3b)" if current != original else ""
        print(f"{_page_name(page)}: the detector found {len(boxes)} boxes{note}")
        image = _labelled_page(folders, page, boxes, [str(i) for i in range(len(boxes))])
        tiles.append((image, f"PDF page {page['pdf_page_number']}: {len(boxes)} boxes"))
        if sources[page["page_id"]] == "manual":
            print("  (boxes from a file you supplied, not from the detector)")
    _show(_sheet(tiles, columns=2, tile_width=520), folders["previews"] / "detector-boxes.png")
    print("Every box has a number. Step 3b checks the boxes against your description from step 2.")


def _corrections(name, value, music, check):
    if not isinstance(value, dict):
        raise NotDone(f"{name} must look like {{2: [...]}}: curly brackets around it.")
    by_number = {page["pdf_page_number"]: page for page in music}
    result = {}
    for number, entries in value.items():
        if number not in by_number:
            raise NotDone(f"{name}: {number!r} is not the number of a music page "
                          f"(music pages: {', '.join(str(n) for n in by_number)}).")
        if not isinstance(entries, (list, tuple)):
            raise NotDone(f"{name}, page {number}: put the entries in square brackets, e.g. {number}: [...].")
        if name == "ADD_BOXES" and len(entries) == 4 and all(isinstance(v, (int, float)) for v in entries):
            entries = [entries]  # a single box without the outer brackets
        result[number] = [check(by_number[number], entry) for entry in entries]
    return result


def _new_box(page, entry):
    if (not isinstance(entry, (list, tuple)) or len(entry) != 4
            or not all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in entry)):
        raise NotDone(f"ADD_BOXES, page {page['pdf_page_number']}: {entry!r} is not a box. "
                      "Write four numbers in round brackets: (left, top, right, bottom).")
    left, top, right, bottom = (round(v) for v in entry)
    if left >= right or top >= bottom:
        raise NotDone(f"ADD_BOXES, page {page['pdf_page_number']}: in {entry!r}, left must be smaller than right "
                      "and top smaller than bottom. The order is (left, top, right, bottom).")
    if left < 0 or top < 0 or right > page["width"] or bottom > page["height"]:
        raise NotDone(f"ADD_BOXES, page {page['pdf_page_number']}: {entry!r} reaches outside the page "
                      f"(the picture is {page['width']} × {page['height']} pixels).")
    return dict(zip(COORDS, (left, top, right, bottom)))


def _overlap(a, b):
    width = min(a["lrx"], b["lrx"]) - max(a["ulx"], b["ulx"])
    height = min(a["lry"], b["lry"]) - max(a["uly"], b["uly"])
    if width <= 0 or height <= 0:
        return 0.0
    area_a = (a["lrx"] - a["ulx"]) * (a["lry"] - a["uly"])
    area_b = (b["lrx"] - b["ulx"]) * (b["lry"] - b["uly"])
    return width * height / (area_a + area_b - width * height)


def _apply_corrections(folders, page, add, remove):
    """Rebuild the boxes in use from the detector's boxes plus ADD_BOXES minus REMOVE_BOXES."""
    path, backup = _box_files(folders, page)
    original, _ = _boxes(folders, page)
    count = len(original["measures"])
    for index in remove:
        if not isinstance(index, int) or isinstance(index, bool) or not 0 <= index < count:
            raise NotDone(f"REMOVE_BOXES, page {page['pdf_page_number']}: {index!r} is not a box number "
                          f"(this page has boxes 0 to {count - 1}).")
    kept = [box for index, box in enumerate(original["measures"]) if index not in remove]
    for box in add:
        for index, other in enumerate(original["measures"]):
            if index not in remove and _overlap(box, other) > 0.5:
                raise NotDone(f"ADD_BOXES, page {page['pdf_page_number']}: {tuple(box.values())} covers "
                              f"most of box {index}. Is that bar really missing?")
    if not add and not remove:
        if backup.is_file():  # no corrections any more: use the detector's boxes again
            path.write_bytes(backup.read_bytes())
            backup.unlink()
        return
    edited = deepcopy(original)
    edited["measures"] = kept + add
    if not backup.is_file():
        backup.write_bytes(path.read_bytes())
    text = json.dumps(edited, indent=2, ensure_ascii=False) + "\n"
    if path.read_text(encoding="utf-8") != text:
        path.write_text(text, encoding="utf-8")


def _ruler_picture(folders, page, boxes, labels, band, path):
    from matplotlib.figure import Figure
    from matplotlib.patches import Rectangle
    from matplotlib.ticker import MultipleLocator

    x0, y0, x1, y1 = band
    with PILImage.open(folders["pages"] / page["image"]) as source:
        crop = source.convert("L").crop(band)
    figure = Figure(figsize=(13, 13 * (y1 - y0) / (x1 - x0) + 1.2), dpi=100)
    axes = figure.add_subplot()
    axes.imshow(crop, cmap="gray", extent=(x0, x1, y1, y0))
    for box, label in zip(boxes, labels):
        color = "tab:blue" if label == "new" else "red"
        axes.add_patch(Rectangle((box["ulx"], box["uly"]), box["lrx"] - box["ulx"], box["lry"] - box["uly"],
                                 fill=False, edgecolor=color, linewidth=1.5))
        axes.text(box["ulx"] + 12, box["uly"] + 12, label, color=color, fontsize=11, va="top",
                  bbox={"facecolor": "white", "edgecolor": "none", "pad": 1})
    axes.set_xlim(x0, x1)
    axes.set_ylim(y1, y0)
    axes.xaxis.set_major_locator(MultipleLocator(100))
    axes.xaxis.set_minor_locator(MultipleLocator(50))
    axes.yaxis.set_major_locator(MultipleLocator(100))
    axes.tick_params(axis="x", labelrotation=90, labelsize=8)
    axes.tick_params(axis="y", labelsize=8)
    axes.xaxis.set_ticks_position("both")
    axes.tick_params(axis="x", labeltop=True)
    axes.grid(True, which="both", color="tab:blue", alpha=0.25, linewidth=0.6)
    axes.set_xlabel("pixels from the left edge  →  left, right")
    axes.set_ylabel("pixels from the top  →  top, bottom")
    figure.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(path)
    _show_file(path)


def _diagnose(folders, music):
    print("⚠ NOT DONE. The boxes do not match your description yet:\n")
    for page in music:
        original, current = _boxes(folders, page)
        rows = _ordered_systems(current, page)
        found = [len(row) for row in rows]
        described = [len(system) for system in page["systems"]]
        if found == described:
            continue
        labels = _labels_for(original, current["measures"])
        if len(found) != len(described):
            print(f"{_page_name(page)}: the boxes form {len(found)} rows, but you described "
                  f"{len(described)} systems. Maybe a whole system has no boxes, or the description "
                  "in step 2 has a system too many or too few.")
            _ruler_picture(folders, page, current["measures"], labels,
                           (0, 0, page["width"], page["height"]),
                           folders["previews"] / f"{page['page_id']}-check.png")
            continue
        for number, (row, system) in enumerate(zip(rows, page["systems"]), 1):
            if len(row) == len(system):
                continue
            difference = len(system) - len(row)
            problem = (f"{difference} box{'es are' if difference > 1 else ' is'} missing" if difference > 0
                       else f"{-difference} box{'es' if difference < -1 else ''} too many")
            print(f"{_page_name(page)}, system {number} (bars {_ranges(list(dict.fromkeys(system)))}): "
                  f"you described {len(system)} bars, the detector drew {len(row)} boxes, so {problem}.")
            boxes = [current["measures"][item["box_index"]] for item in row]
            top = max(0, min(box["uly"] for box in boxes) - 150)
            bottom = min(page["height"], max(box["lry"] for box in boxes) + 150)
            _ruler_picture(folders, page, boxes, [labels[item["box_index"]] for item in row],
                           (0, top, page["width"], bottom),
                           folders["previews"] / f"{page['page_id']}-system{number}-check.png")
            print("Boxes on this system, as (left, top, right, bottom):")
            for item in row:
                box = current["measures"][item["box_index"]]
                print(f"  box {labels[item['box_index']]:>3}: ({', '.join(str(round(box[k])) for k in COORDS)})")
            print()
    print("How to fix it:")
    print("  • A box is missing: add it to ADD_BOXES (see the instructions above this cell).")
    print("  • A box is too many: put its number into REMOVE_BOXES.")
    print("  • Or the description in step 2 is wrong: correct it there and run step 2 again.")
    print("Then run this cell again.")


@_step
def check_bars(output_folder, add_boxes=None, remove_boxes=None):
    """Step 3b: apply your corrections, compare the boxes with step 2, and number them."""
    folders = _folders(output_folder)
    music = _music_pages(folders)
    if any(not _box_files(folders, page)[0].is_file() for page in music):
        raise NotDone("Some music pages have no boxes yet. Run step 3a first.")
    add = _corrections("ADD_BOXES", add_boxes or {}, music, _new_box)
    remove = _corrections("REMOVE_BOXES", remove_boxes or {}, music, lambda page, index: index)
    for page in music:
        _apply_corrections(folders, page, add.get(page["pdf_page_number"], []),
                           remove.get(page["pdf_page_number"], []))
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            build_measure_alignment(folders["pages_json"], output=folders["alignment"])
    except ValueError as problem:
        if not str(problem).startswith("System count gate failed"):
            raise
        folders["alignment"].unlink(missing_ok=True)
        _diagnose(folders, music)
        return
    regions = json.loads(folders["alignment"].read_text(encoding="utf-8"))["regions"]
    changes = sum(len(v) for v in add.values()) + sum(len(v) for v in remove.values())
    if changes:
        print(f"Your corrections: {sum(len(v) for v in add.values())} box(es) added, "
              f"{sum(len(v) for v in remove.values())} removed.")
    print(f"✓ Every bar has a box: {len(regions)} boxes on {len(music)} pages.")
    print("Look at every picture: does each bar number (movement/bar) sit on the right bar?\n")
    for path in draw_alignment_overlays(folders["alignment"], folders["overlays"]):
        with PILImage.open(path) as overlay:
            preview = overlay.resize((1100, round(overlay.height * 1100 / overlay.width)))
        _show(preview, folders["previews"] / f"check-{path.name}")
    print("If a number sits on the wrong bar, the description in step 2 does not match the print. "
          "Correct it there and run steps 2, 3a and 3b again.")


# ---------- Step 4 ----------

def _number_movement(text):
    match = re.search(r"<mdiv\b[^>]*?(/?)>", text)
    return text[:match.start()] + match[0][:-len(match[1]) - 1] + ' n="1"' + match[1] + ">" + text[match.end():]


def _result_path(mei_file, output_folder):
    return _folders(output_folder)["mei"] / f"{resolve_repo_path(mei_file).stem}_with_facsimile.mei"


@_step
def merge_with_mei(mei_file, output_folder, *, checked_by="", match_line_breaks=True):
    """Step 4: write a new MEI file with a <facsimile> and measure/@facs links."""
    match_line_breaks = _switch(match_line_breaks, "MATCH_LINE_BREAKS_TO_PDF")
    if not isinstance(checked_by, str):
        raise NotDone('YOUR_NAME must be text in quotes, for example "Clara Schumann".')
    folders = _folders(output_folder)
    output = _result_path(mei_file, output_folder)
    output.unlink(missing_ok=True)  # step 5 must not show an older result if this run fails
    _music_pages(folders)
    try:  # check the boxes as they are now, not an older result
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            build_measure_alignment(folders["pages_json"], output=folders["alignment"])
    except (ValueError, FileNotFoundError):
        raise NotDone("Step 3 is not finished: the boxes do not match your description yet. Go back to step 3b.")
    if checked_by.strip():
        data = json.loads(folders["alignment"].read_text(encoding="utf-8"))
        changed = False
        for entry in data["review"].values():
            if entry.get("status") != "approved" or entry.get("reviewer") != checked_by.strip():
                entry.update(status="approved", reviewer=checked_by.strip(), date=date.today().isoformat())
                changed = True
        if changed:
            folders["alignment"].write_text(json.dumps(data, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

    mei = resolve_repo_path(mei_file)
    mdiv_n, _ = _mei_movement(mei)
    text = mei.read_bytes().decode("utf-8")
    if mdiv_n is None:
        text = _number_movement(text)
    output.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="camat-guided-") as temp:
        staged = Path(temp) / mei.name
        staged.write_bytes(text.encode("utf-8"))
        try:
            report = integrate_pdf_facsimile(staged, folders["pages_json"], folders["alignment"], output)
        except ValueError as problem:
            if "gate failed" in str(problem):
                raise NotDone(f"The MEI and your description from step 2 do not agree: {problem}")
            raise
    breaks = write_page_breaks(output, folders["pages_json"], output) if match_line_breaks else None

    print(f"✓ Done. {report['measures_linked']} bars of the MEI now point to "
          f"{report['zones']} boxes on {report['surfaces']} pages.")
    print(f"  New file:      {display_path(output)}")
    print(f"  Page pictures: {display_path(folders['pages'])} (the new file refers to them; keep the folders together)")
    if breaks:
        print(f"  Line breaks:   {breaks['pb']} page breaks and {breaks['sb']} system breaks now follow the PDF.")
    elif report["pb_not_at_page_start"]:
        print(f"  Line breaks:   {len(report['pb_not_at_page_start'])} page break(s) of the MEI do not match the PDF; "
              "set MATCH_LINE_BREAKS_TO_PDF = True to replace them.")
    if report["unreviewed_pages"]:
        print("  ⚠ YOUR_NAME is empty, so the file records that nobody has checked the boxes. "
              "Write your name in the settings cell once you have checked the pictures of step 3b, "
              "run that cell and this one again.")
    else:
        review = json.loads(folders["alignment"].read_text(encoding="utf-8"))["review"].values()
        names = sorted({f"{entry['reviewer']} ({entry['date']})" for entry in review})
        print(f"  Checked by:    {', '.join(names)}")
    if report["other_facsimiles"]:
        print(f"  The MEI already had {report['other_facsimiles']} other <facsimile>; it was kept.")

    result = output.read_text(encoding="utf-8")
    excerpt = [re.search(r"<surface\b[^>]*>", result), re.search(r"<graphic\b[^>]*>", result),
               *list(re.finditer(r"<zone\b[^>]*>", result))[:2]]
    measure = re.search(r"<measure\b[^>]*\bfacs=[^>]*>", result)
    print("\nWhat was added (first lines):")
    for index, match in enumerate(excerpt):
        print(("   " if index else "") + match[0])
    print("   ...")
    print(measure[0])


# ---------- Step 5 ----------

@_step
def open_result(mei_file, output_folder):
    """Step 5: show the new file in CAMAT's facsimile viewer."""
    path = _result_path(mei_file, output_folder)
    if not path.is_file():
        raise NotDone("There is no result yet. Run step 4 first.")
    if not _in_jupyter():
        print(f"Result: {display_path(path)} (open this notebook in Jupyter to see the viewer)")
        return None
    return launch_interactive_facsimile_viewer(str(path), verovio_breaks="encoded")


print("✓ Helpers are ready.")

## ✏️ Your settings

The next cell holds all settings. For the example:

1. Write your name between the quotes of `YOUR_NAME`.
2. Set `DOWNLOAD_EXAMPLE_PDF` and `SEND_PAGES_TO_DETECTOR` to `True` once you have read what they do (steps 1 and 3 explain it).
3. Run the cell.

**Run this cell again after every change.** The steps below use what this cell last stored, not what you see in it.

In [ ]:
# ✏️ Your settings. Change the text between quotes and True / False only.

# The MEI file (the encoded music). The example file comes with CAMAT.
MEI_FILE = "test_corpus/Buxtehude-DdT11-app01-suite.mei"

# The PDF (the scanned score). Step 1 explains how to get the example PDF.
PDF_FILE = "converted_mei/pdf_guided/bsb00023199_0170-0173.pdf"

# Your name: stored in the result as the person who checked the bar boxes.
YOUR_NAME = ""

# True = yes, False = no.
DOWNLOAD_EXAMPLE_PDF = False      # download the example PDF from the Bayerische Staatsbibliothek
SEND_PAGES_TO_DETECTOR = False    # upload the music pages to the measure detector (needs internet)
MATCH_LINE_BREAKS_TO_PDF = True   # page and line breaks of the result follow the PDF

# Where the results go. You can leave this as it is.
OUTPUT_FOLDER = "converted_mei/pdf_guided"

## Step 1 — Turn the PDF into page images

The computer cannot measure bars inside a PDF directly, so CAMAT first turns every PDF page into a picture (PNG, 300 dots per inch). Each picture keeps the number of its PDF page: the first page of the file is **PDF page 1**, even when the print says "160" on it.

**Get the example PDF** in one of two ways. The scans belong to the Bayerische Staatsbibliothek (BSB) in Munich, under the licence [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/): non-commercial use, name the source, share changed versions under the same licence.

- **Automatically:** set `DOWNLOAD_EXAMPLE_PDF = True` in the settings cell. This answers "yes" for you to the BSB's question whether you have read its rights notice and agree to it.
- **By hand:** open the [BSB download form for images 170–173](https://download.digitale-sammlungen.de/BOOKS/download.pl?id=00023199&ersteseite=170&letzteseite=173). At *"Ich versichere, den Rechtehinweis gelesen zu haben und bin damit einverstanden"* ("I confirm that I have read the rights notice and agree to it") choose **Ja**, click the **WEITER** button next to *PDF*, and save the PDF linked on the next page. Put it into the folder `converted_mei/pdf_guided/` (create the folder if needed) under the name `bsb00023199_0170-0173.pdf`, or set `PDF_FILE` to where you saved it.

The PDF has **5 pages**: page 1 is a cover sheet from the library; pages 2–5 are the printed pages 160–163. Keep the file: a new download of the same pages counts as a different file.

This step needs the free program **Poppler**. If it is missing, the cell says how to install it.

In [ ]:
make_page_images(PDF_FILE, OUTPUT_FOLDER, download_example=DOWNLOAD_EXAMPLE_PDF)

## Step 2 — ✏️ Describe the music pages

Look at the page pictures from step 1. For every page **with music**, write one line in the cell below:

```
PDF page | printed page | bars on each system, top to bottom
```

- A **system** is one line of music: all staves that sound together, joined by a brace or bracket on the left. Page 160 of the example has three systems.
- Write the bars of a system as a range like `1-4`, and separate the systems with commas.
- Use the **bar numbers of the MEI file** (`<measure n="…">`). This 1903 print has no bar numbers, so count. An upbeat counts as a bar when the MEI encodes it as its own `<measure>`: in the example, bar 10 is the short upbeat after the repeat sign at the start of the third system on page 160.
- A bar that starts at the end of one system and continues on the next gets its number twice: `10-14, 14-18`.
- Pages you do not list count as pages without music. You may name them, for example `1 | cover` (other words: title, preface, blank, advertisement, duplicate).
- Lines that start with `#` are ignored.

The example's description is already filled in. The cell compares it with the MEI file and tells you if bars are missing, extra, or in a different order.

In [ ]:
# ✏️ One line per page:  PDF page | printed page | bars on each system, top to bottom
MUSIC_PAGES = """
1 | cover
2 | 160 | 1-4, 5-9, 10-14
3 | 161 | 15-18, 19-25, 26-33, 34-40
4 | 162 | 41-47, 48-55, 56-63, 64-70
5 | 163 | 71-74, 75-79, 80-84, 85-89
"""

describe_pages(MUSIC_PAGES, MEI_FILE, OUTPUT_FOLDER)

## Step 3 — Find the bars on the pages

In MEI, a **zone** is a rectangle on a page picture. Four numbers describe it, counted in pixels from the top-left corner of the picture: **left**, **top**, **right**, **bottom**. In step 4, every bar of the MEI will point to its zone.

Drawing 89 rectangles by hand takes long, so a **measure detector** does it: a neural network of the Edirom project, at `measure-detector.edirom.de`. It looks only at the picture, not at the MEI, and returns one box per bar it recognises. It is good but not perfect, so you check its work in step 3b.

### 3a — Let the detector find the bars

With `SEND_PAGES_TO_DETECTOR = True`, the pictures of the music pages are uploaded to the detector. Its answers are saved in your output folder, so running this cell again uploads nothing. Every box gets a small number; you need these numbers only to remove a wrong box.

In [ ]:
find_bars(OUTPUT_FOLDER, send_to_detector=SEND_PAGES_TO_DETECTOR)

### 3b — ✏️ Check the boxes and add the ones the detector missed

The cell below compares the boxes with your description from step 2, system by system.

- **If they match,** it labels every box with its bar number (movement/bar, for example `1/10`) and shows the pages. Check that every label sits on the right bar.
- **If they do not match,** it says which system is wrong and shows that system enlarged, with rulers in pixels.

**Exercise: add a missing box yourself.** The detector misses **bar 10** on PDF page 2, the short upbeat after the repeat sign at the start of the third system. Run the cell once as it is and look at the enlarged picture. Then work out the four numbers of the missing box:

1. **top** and **bottom**: copy them from the neighbouring box in the list under the picture. The new box covers the same staves.
2. **left**: where the staff lines begin. Read it on the ruler.
3. **right**: just after the first barline. The box may overlap the next box a little; the detector's boxes overlap too.

Write them into `ADD_BOXES` in the order (left, top, right, bottom): remove the `#` in front of the prepared line, put in your numbers, and run the cell again. Small differences do not matter.

<details>
<summary>Solution (open it only after trying)</summary>

```python
ADD_BOXES = {
    2: [(180, 2540, 530, 3560)],
}
```

Anything from about 150 to 250 for *left* and 500 to 560 for *right* works.
</details>

A box around something that is not a bar (a title, say) goes into `REMOVE_BOXES`, with the number shown in the pictures of step 3a: `{3: [7]}` removes box 7 on PDF page 3. To undo a correction, delete it and run the cell again.

In [ ]:
# ✏️ Boxes to add:     {PDF page: [(left, top, right, bottom), ...]}
ADD_BOXES = {
    # 2: [(left, top, right, bottom)],
}

# ✏️ Boxes to remove:  {PDF page: [box number, ...]}
REMOVE_BOXES = {
}

check_bars(OUTPUT_FOLDER, ADD_BOXES, REMOVE_BOXES)

## Step 4 — Merge the boxes into the MEI

This step writes a **new** MEI file; your original stays as it is. CAMAT adds a `<facsimile>` section with one `<surface>` per music page. A surface holds a `<graphic>` (the page picture) and one `<zone>` per box. Then every `<measure>` gets a `facs` attribute that points to its zone:

```xml
<surface xml:id="surface-…-p0002" n="160" …>
   <graphic target="../pages/page-0002.png" …/>
   <zone xml:id="zone-…-p0002-001" type="measure" ulx="554" uly="601" lrx="944" lry="1578" …/>
   …
<measure n="1" facs="#zone-…-p0002-001" …>
```

- `YOUR_NAME` is stored as the person who checked the boxes in step 3b. If it is empty, the cell reminds you.
- With `MATCH_LINE_BREAKS_TO_PDF = True`, the page breaks (`<pb>`) and system breaks (`<sb>`) of the new file follow the PDF, so the score in step 5 breaks its lines where the print does.
- The new file finds the page pictures by a relative path (`../pages/…`). Keep the output folder together when you copy it.
- Running this step again replaces the new file. If you edit the new file by hand, copy it somewhere else first.

In [ ]:
merge_with_mei(MEI_FILE, OUTPUT_FOLDER, checked_by=YOUR_NAME, match_line_breaks=MATCH_LINE_BREAKS_TO_PDF)

## Step 5 — Look at the result

The viewer shows the encoded score (drawn by Verovio) next to the page pictures. Move the mouse over a bar on either side and its partner lights up. Use the toolbar to turn pages and zoom. The viewer only reads the file.

To open the result again later, use [`mei_facsimile_viewer.ipynb`](mei_facsimile_viewer.ipynb) and set `MEI_SOURCE` to the new file printed in step 4.

In [ ]:
viewer = open_result(MEI_FILE, OUTPUT_FOLDER)

## Your own score

1. In the settings cell, set `MEI_FILE` to your MEI file (one movement) and `PDF_FILE` to your PDF, for example `"notebooks/my_score.pdf"` if you put the PDF next to this notebook. Set `DOWNLOAD_EXAMPLE_PDF = False`.
2. Give every score its own `OUTPUT_FOLDER`, for example `"converted_mei/pdf_guided_my_score"`. Run the settings cell.
3. Run step 1 and look at the pictures.
4. Replace the example lines in step 2 with your pages, and run step 2 until it shows ✓.
5. Run step 3a. In step 3b, empty `ADD_BOXES` and `REMOVE_BOXES` first, then fix what the cell reports.
6. Run steps 4 and 5.

## If something goes wrong

| What you see | What to do |
| --- | --- |
| `NameError: name '…' is not defined` | Run the cells from the top: setup, helpers, settings, then the steps in order. |
| You changed a setting, but nothing changed | Run the settings cell again after every change. |
| `SyntaxError` in a ✏️ cell | A quote, comma or bracket is missing. Compare with the example lines. |
| Step 1: "needs the free program Poppler" | Install it as the message says, restart the kernel, run again from the top. |
| Step 1: "already holds pictures of another PDF" | You changed the PDF (a new download also counts). Choose a new `OUTPUT_FOLDER`. |
| Step 2: bars missing or extra | Compare your lines with the print and with the bar numbers in the MEI. |
| Step 3b: a box is missing or too many | Do the exercise in step 3b with your own numbers. |
| Step 3b: a label sits on the wrong bar | Your description in step 2 does not match the print. Correct it, then run steps 2, 3a and 3b again. |